# Predict Customer Churn - CatBoost Baseline

This notebook trains a first baseline model, checks ROC-AUC on a holdout split, and writes a Kaggle submission file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

TARGET_COLUMN = 'Churn'
ID_COLUMN = 'id'
CAT_FEATURES = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]
NUM_FEATURES = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

TRAIN_PATH = Path('data/train.csv')
TEST_PATH = Path('data/test.csv')
OUTPUT_DIR = Path('outputs')
ARTIFACT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

y = train_df[TARGET_COLUMN].map({'No': 0, 'Yes': 1}).astype('int8')
X = train_df[CAT_FEATURES + NUM_FEATURES].copy()
X_test = test_df[CAT_FEATURES + NUM_FEATURES].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_feature_indices = [X.columns.get_loc(col) for col in CAT_FEATURES]

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('positive rate:', y.mean())

In [ ]:
model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    early_stopping_rounds=200,
    random_seed=42,
    task_type='CPU',
    verbose=200,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

valid_pred = model.predict_proba(X_valid)[:, 1]
roc_auc_valid = roc_auc_score(y_valid, valid_pred)
best_iteration = model.get_best_iteration()
print('Validation ROC-AUC:', round(float(roc_auc_valid), 6))
print('Best iteration:', best_iteration)

In [ ]:
best_iteration = int(best_iteration) if best_iteration and best_iteration > 0 else 2000
final_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    iterations=best_iteration,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    task_type='CPU',
    verbose=200,
)

final_model.fit(X, y, cat_features=cat_feature_indices)
test_pred = final_model.predict_proba(X_test)[:, 1]

assert np.all(test_pred >= 0.0) and np.all(test_pred <= 1.0), 'Probabilities out of bounds'

submission_df = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN], TARGET_COLUMN: test_pred})
assert list(submission_df.columns) == ['id', 'Churn']
assert len(submission_df) == len(test_df)

submission_path = OUTPUT_DIR / 'submission_baseline.csv'
model_path = ARTIFACT_DIR / 'catboost_baseline.cbm'
submission_df.to_csv(submission_path, index=False)
final_model.save_model(str(model_path))

print('Saved:', submission_path)
print('Saved:', model_path)
submission_df.head()

## Optional CLI submission from container

Run this in a terminal after setting `KAGGLE_COMPETITION` and mounting `kaggle.json`:

`kaggle competitions submit -c $KAGGLE_COMPETITION -f outputs/submission_baseline.csv -m "catboost baseline v1"`